# Baseline Models — Telco Churn
DummyClassifier e LogisticRegression com StratifiedKFold + MLflow tracking.

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
import warnings
warnings.filterwarnings("ignore")

import mlflow
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

from src.preprocessing import build_preprocessor, _clean, TARGET

logging.basicConfig(level=logging.INFO)
SEED = 42
DATA_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

In [ ]:
df = pd.read_csv(DATA_PATH)
df = _clean(df)

X = df.drop(columns=[TARGET])
y = df[TARGET].values
print(f"Shape: {X.shape} | Churn rate: {y.mean():.2%}")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = ["f1", "roc_auc", "precision", "recall"]

def run_cv(name, estimator):
    """Run cross-validation and log results to MLflow."""
    pipe = Pipeline([("pre", build_preprocessor()), ("clf", estimator)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=SCORING, n_jobs=-1)
    return {k: scores[f"test_{k}"].mean() for k in SCORING}

In [ ]:
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("churn-baselines")

baselines = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent", random_state=SEED),
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
}

results = {}
for name, clf in baselines.items():
    with mlflow.start_run(run_name=name):
        mlflow.log_param("model", name)
        mlflow.log_param("cv_folds", 5)
        mlflow.log_param("seed", SEED)
        mlflow.log_param("dataset", "WA_Fn-UseC_-Telco-Customer-Churn.csv")

        metrics = run_cv(name, clf)
        mlflow.log_metrics(metrics)
        results[name] = metrics
        print(f"[{name}] {metrics}")

In [ ]:
pd.DataFrame(results).T.round(4)